# Install Libraries

In [ ]:
!pip install pandas numpy faker tqdm -q
print("Libraries installed")


# PharmaFlow Data Generation

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
from tqdm import tqdm
import os
import random

# fixed seed so I get the same numbers every time I run this
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
Faker.seed(SEED)

fake = Faker('de_DE')

# where the CSVs will land
OUT = "pharmaflow_data"
os.makedirs(OUT, exist_ok=True)

print("Starting data generation...")
print(f"Output folder: {OUT}/")

# ---- volume settings ----
N_PATIENTS         = 400_000
N_PRODUCTS         = 15_000
N_ORDERS           = 2_000_000
N_PRESCRIPTIONS    = 900_000     # about 45% of orders will be Rx
RETURN_RATE_OTC    = 0.08        # 8% of OTC orders will be returned
DATE_START         = datetime(2023, 1, 1)
DATE_END           = datetime(2025, 12, 31)


# ====================
# 1. FULFILLMENT CENTERS
# ====================
print("\n[1/9] Fulfillment centers...")

fulfillment_centers = pd.DataFrame([
    {"fc_id": "FC001", "fc_name": "Sevenum",  "country": "NL", "city": "Sevenum",  "capacity_orders_per_day": 80000, "cold_chain_enabled": True,  "opened_year": 2017},
    {"fc_id": "FC002", "fc_name": "Köln",     "country": "DE", "city": "Köln",     "capacity_orders_per_day": 50000, "cold_chain_enabled": True,  "opened_year": 2019},
    {"fc_id": "FC003", "fc_name": "München",  "country": "DE", "city": "München",  "capacity_orders_per_day": 30000, "cold_chain_enabled": False, "opened_year": 2021},
    {"fc_id": "FC004", "fc_name": "Wien",     "country": "AT", "city": "Wien",     "capacity_orders_per_day": 15000, "cold_chain_enabled": True,  "opened_year": 2022},
    {"fc_id": "FC005", "fc_name": "Zürich",   "country": "CH", "city": "Zürich",   "capacity_orders_per_day": 10000, "cold_chain_enabled": False, "opened_year": 2023},
])
fulfillment_centers.to_csv(f"{OUT}/fulfillment_centers.csv", index=False)
print(f"   {len(fulfillment_centers)} fulfillment centers")


# ====================
# 2. INSURANCE PROVIDERS
# ====================
print("\n[2/9] Insurance providers...")

insurance_providers = pd.DataFrame([
    {"insurance_id": "INS01", "insurance_name": "TK (Techniker Krankenkasse)", "type": "Gesetzlich", "country": "DE", "market_share_pct": 14.0},
    {"insurance_id": "INS02", "insurance_name": "AOK Bundesverband",           "type": "Gesetzlich", "country": "DE", "market_share_pct": 36.0},
    {"insurance_id": "INS03", "insurance_name": "Barmer",                       "type": "Gesetzlich", "country": "DE", "market_share_pct": 11.0},
    {"insurance_id": "INS04", "insurance_name": "DAK-Gesundheit",               "type": "Gesetzlich", "country": "DE", "market_share_pct":  7.0},
    {"insurance_id": "INS05", "insurance_name": "IKK Classic",                  "type": "Gesetzlich", "country": "DE", "market_share_pct":  4.0},
    {"insurance_id": "INS06", "insurance_name": "Debeka",                       "type": "Privat",     "country": "DE", "market_share_pct":  3.5},
    {"insurance_id": "INS07", "insurance_name": "Allianz Private Krankenvers.", "type": "Privat",     "country": "DE", "market_share_pct":  3.0},
    {"insurance_id": "INS08", "insurance_name": "AXA Krankenversicherung",      "type": "Privat",     "country": "DE", "market_share_pct":  2.5},
    {"insurance_id": "INS09", "insurance_name": "ÖGK",                          "type": "Gesetzlich", "country": "AT", "market_share_pct":  6.0},
    {"insurance_id": "INS10", "insurance_name": "SVS",                          "type": "Gesetzlich", "country": "AT", "market_share_pct":  2.0},
    {"insurance_id": "INS11", "insurance_name": "Helsana",                      "type": "Gesetzlich", "country": "CH", "market_share_pct":  6.5},
    {"insurance_id": "INS12", "insurance_name": "CSS Versicherung",             "type": "Gesetzlich", "country": "CH", "market_share_pct":  4.5},
])
insurance_providers.to_csv(f"{OUT}/insurance.csv", index=False)
print(f"   {len(insurance_providers)} insurance providers")


# ====================
# 3. PATIENTS
# ====================
print(f"\n[3/9] Patients ({N_PATIENTS:,})...")

# split patients across DACH countries (most in Germany)
country_choices = np.random.choice(
    ["DE", "AT", "CH", "NL"],
    size=N_PATIENTS,
    p=[0.72, 0.10, 0.08, 0.10]
)

# city weights by population (Berlin biggest, Essen smallest in DE)
city_map = {
    "DE": {
        "Berlin":     0.22,
        "Hamburg":    0.14,
        "München":    0.13,
        "Köln":       0.09,
        "Frankfurt":  0.09,
        "Stuttgart":  0.08,
        "Düsseldorf": 0.07,
        "Leipzig":    0.06,
        "Dortmund":   0.06,
        "Essen":      0.06,
    },
    "AT": {
        "Wien":       0.55,
        "Graz":       0.15,
        "Linz":       0.12,
        "Salzburg":   0.10,
        "Innsbruck":  0.08,
    },
    "CH": {
        "Zürich":     0.35,
        "Genf":       0.22,
        "Basel":      0.18,
        "Bern":       0.15,
        "Lausanne":   0.10,
    },
    "NL": {
        "Amsterdam":  0.32,
        "Rotterdam":  0.24,
        "Den Haag":   0.20,
        "Utrecht":    0.14,
        "Eindhoven":  0.10,
    },
}

# basic patient details
patient_ids = np.arange(1, N_PATIENTS + 1)
signup_days = np.random.randint(0, (DATE_END - DATE_START).days, size=N_PATIENTS)
signup_dates = [DATE_START + timedelta(days=int(d)) for d in signup_days]

ages = np.clip(np.random.normal(48, 16, N_PATIENTS).astype(int), 18, 95)
genders = np.random.choice(["F", "M", "D"], size=N_PATIENTS, p=[0.54, 0.45, 0.01])

# assign insurance provider based on which country the patient is in
ins_de = ["INS01","INS02","INS03","INS04","INS05","INS06","INS07","INS08"]
ins_at = ["INS09","INS10"]
ins_ch = ["INS11","INS12"]
ins_nl = ["INS01"]  # placeholder

insurance_ids = []
for c in country_choices:
    if c == "DE":  insurance_ids.append(np.random.choice(ins_de))
    elif c == "AT": insurance_ids.append(np.random.choice(ins_at))
    elif c == "CH": insurance_ids.append(np.random.choice(ins_ch))
    else:           insurance_ids.append(np.random.choice(ins_nl))

# pick a city for each patient, weighted by city size
cities = np.empty(N_PATIENTS, dtype=object)
for country_code, city_weights in city_map.items():
    mask = country_choices == country_code
    n_country = mask.sum()
    if n_country == 0:
        continue
    cities[mask] = np.random.choice(
        list(city_weights.keys()),
        size=n_country,
        p=list(city_weights.values())
    )

# 32% of patients have a chronic condition (they reorder more)
has_chronic = np.random.choice([True, False], size=N_PATIENTS, p=[0.32, 0.68])

patients = pd.DataFrame({
    "patient_id": patient_ids,
    "signup_date": signup_dates,
    "age": ages,
    "gender": genders,
    "country": country_choices,
    "city": cities,
    "insurance_id": insurance_ids,
    "has_chronic_condition": has_chronic,
})
patients.to_csv(f"{OUT}/patients.csv", index=False)
print(f"   {len(patients):,} patients")


# ====================
# 4. PRODUCTS
# ====================
print(f"\n[4/9] Products ({N_PRODUCTS:,})...")

# each category has: (weight in product mix, % that are Rx, price range, % that need cold chain)
category_config = {
    "Prescription_Cardio":     (0.08, 1.00,  8,  80, 0.05),
    "Prescription_Diabetes":   (0.06, 1.00, 12, 120, 0.35),   # insulin needs cold chain
    "Prescription_Respiratory":(0.05, 1.00,  9,  60, 0.02),
    "Prescription_Mental_Health":(0.04,1.00, 10,  90, 0.00),
    "Prescription_Hormone":    (0.03, 1.00, 15, 110, 0.10),
    "OTC_Pain_Relief":         (0.12, 0.00,  3,  20, 0.00),
    "OTC_Cold_Flu":            (0.10, 0.00,  4,  25, 0.00),
    "OTC_Digestive":           (0.07, 0.00,  4,  30, 0.00),
    "Vitamins_Supplements":    (0.15, 0.00,  5,  60, 0.00),
    "Personal_Care":           (0.13, 0.00,  3,  45, 0.00),
    "Baby_Mother_Care":        (0.08, 0.00,  4,  50, 0.00),
    "Medical_Devices":         (0.09, 0.00, 15, 250, 0.00),
}

products_list = []
for prod_id in range(1, N_PRODUCTS + 1):
    cat = np.random.choice(
        list(category_config.keys()),
        p=[v[0] for v in category_config.values()]
    )
    weight, rx_pct, p_min, p_max, cc_pct = category_config[cat]

    is_rx = np.random.random() < rx_pct
    price = round(np.random.uniform(p_min, p_max), 2)
    cold_chain = np.random.random() < cc_pct

    products_list.append({
        "product_id": prod_id,
        "product_name": f"{cat.replace('_',' ')} Item {prod_id}",
        "category": cat,
        "is_prescription": is_rx,
        "requires_cold_chain": cold_chain,
        "unit_price_eur": price,
        "manufacturer": fake.company(),
    })

products = pd.DataFrame(products_list)
products.to_csv(f"{OUT}/products.csv", index=False)
print(f"   {len(products):,} products")
print(f"      Rx products: {products['is_prescription'].sum():,}")
print(f"      OTC products: {(~products['is_prescription']).sum():,}")
print(f"      Cold-chain products: {products['requires_cold_chain'].sum():,}")


# ====================
# 5. ORDERS
# ====================
print(f"\n[5/9] Orders ({N_ORDERS:,}) - this takes about 2 minutes...")

# Q4 is the peak season for pharmacies (cold/flu, year-end vitamin restocks)
total_days = (DATE_END - DATE_START).days
order_days = np.random.randint(0, total_days, size=N_ORDERS)

# bias toward winter months
month_weights = np.array([1.3, 1.25, 1.1, 0.9, 0.85, 0.8, 0.8, 0.85, 0.95, 1.1, 1.25, 1.4])
order_dates = []
for d in tqdm(order_days, desc="   dates"):
    base_date = DATE_START + timedelta(days=int(d))
    # keep this date only with probability proportional to its month weight
    if np.random.random() < month_weights[base_date.month - 1] / month_weights.max():
        order_dates.append(base_date)
    else:
        # re-roll into a higher-weight month
        order_dates.append(DATE_START + timedelta(days=int(np.random.randint(0, total_days))))

order_ids = np.arange(1, N_ORDERS + 1)

# 60% of orders go to chronic patients because they reorder regularly
chronic_patients = patients[patients["has_chronic_condition"]]["patient_id"].values
non_chronic_patients = patients[~patients["has_chronic_condition"]]["patient_id"].values

chronic_order_mask = np.random.random(N_ORDERS) < 0.60
patient_ids_for_orders = np.where(
    chronic_order_mask,
    np.random.choice(chronic_patients, size=N_ORDERS),
    np.random.choice(non_chronic_patients, size=N_ORDERS)
)

# how the order was placed
channels = np.random.choice(
    ["Web", "Mobile_App", "Phone"],
    size=N_ORDERS,
    p=[0.55, 0.42, 0.03]
)

# which FC fulfills the order (Sevenum handles most volume)
fc_ids = np.random.choice(
    ["FC001","FC002","FC003","FC004","FC005"],
    size=N_ORDERS,
    p=[0.45, 0.28, 0.15, 0.08, 0.04]
)

# about 45% of orders include prescription items
is_rx_order = np.random.random(N_ORDERS) < 0.45

# E-Rezept adoption ramped up after the Jan 2024 mandate in Germany
# 2023: 5%, 2024: 60%, 2025: 92%
e_rezept_flag = []
for i, dt in enumerate(order_dates):
    if not is_rx_order[i]:
        e_rezept_flag.append(False)
    else:
        if dt.year == 2023:   prob = 0.05
        elif dt.year == 2024: prob = 0.60
        else:                 prob = 0.92
        e_rezept_flag.append(np.random.random() < prob)

orders = pd.DataFrame({
    "order_id": order_ids,
    "patient_id": patient_ids_for_orders,
    "order_date": order_dates,
    "fc_id": fc_ids,
    "channel": channels,
    "is_prescription_order": is_rx_order,
    "uses_e_rezept": e_rezept_flag,
    "order_status": np.random.choice(
        ["Delivered", "Delivered", "Delivered", "Delivered", "Cancelled", "Returned"],
        size=N_ORDERS,
        p=[0.88, 0.04, 0.03, 0.02, 0.02, 0.01]
    ),
})
orders.to_csv(f"{OUT}/orders.csv", index=False)
print(f"   {len(orders):,} orders")
print(f"      Rx orders: {orders['is_prescription_order'].sum():,}")
print(f"      E-Rezept orders: {orders['uses_e_rezept'].sum():,}")


# ====================
# 6. ORDER ITEMS
# ====================
print(f"\n[6/9] Order items (~4.5M) - this takes about 3 minutes...")

rx_product_ids = products[products["is_prescription"]]["product_id"].values
otc_product_ids = products[~products["is_prescription"]]["product_id"].values
product_prices = dict(zip(products["product_id"], products["unit_price_eur"]))

# Rx orders tend to be small (avg 2 items), OTC baskets are bigger
items_per_order_rx = np.random.choice([1,2,3,4], size=is_rx_order.sum(), p=[0.45,0.35,0.15,0.05])
items_per_order_otc = np.random.choice([1,2,3,4,5,6,7,8], size=(~is_rx_order).sum(), p=[0.15,0.22,0.20,0.15,0.12,0.08,0.05,0.03])

items_per_order = np.zeros(N_ORDERS, dtype=int)
items_per_order[is_rx_order] = items_per_order_rx
items_per_order[~is_rx_order] = items_per_order_otc

# expand each order into N line items
all_order_ids = np.repeat(order_ids, items_per_order)
all_is_rx = np.repeat(is_rx_order, items_per_order)

total_items = len(all_order_ids)
print(f"   Building {total_items:,} line items...")

# pick a product for each line item (Rx orders get Rx products only)
chosen_products = np.where(
    all_is_rx,
    np.random.choice(rx_product_ids, size=total_items),
    np.random.choice(otc_product_ids, size=total_items)
)

quantities = np.random.choice([1,2,3], size=total_items, p=[0.75, 0.20, 0.05])

# look up the price for each chosen product
unit_prices = np.array([product_prices[p] for p in tqdm(chosen_products, desc="   prices")])
line_totals = np.round(unit_prices * quantities, 2)

# small chance of a discount on a line item
discount_pct = np.random.choice([0, 0, 0, 0.05, 0.10, 0.15], size=total_items, p=[0.70, 0.10, 0.05, 0.08, 0.05, 0.02])
final_line_totals = np.round(line_totals * (1 - discount_pct), 2)

order_items = pd.DataFrame({
    "order_item_id": np.arange(1, total_items + 1),
    "order_id": all_order_ids,
    "product_id": chosen_products,
    "quantity": quantities,
    "unit_price_eur": unit_prices,
    "discount_pct": discount_pct,
    "line_total_eur": final_line_totals,
})
order_items.to_csv(f"{OUT}/order_items.csv", index=False)
print(f"   {len(order_items):,} order items")


# ====================
# 7. PRESCRIPTIONS
# ====================
print(f"\n[7/9] Prescriptions...")

# only Rx orders need a prescription record
rx_orders_df = orders[orders["is_prescription_order"]].copy()
n_rx = len(rx_orders_df)

# need to convert this column to proper datetime to do the date math below
rx_orders_df["order_date"] = pd.to_datetime(rx_orders_df["order_date"])

# doctor writes the prescription 0-14 days before the patient places the order
random_offsets = np.random.randint(0, 14, size=n_rx)
prescription_dates = rx_orders_df["order_date"].values - pd.to_timedelta(random_offsets, unit="D")

# E-Rezept orders get a digital token, paper prescriptions don't
uses_e_rezept = rx_orders_df["uses_e_rezept"].values
e_rezept_tokens = [
    f"ER-{fake.bothify('??######??')}" if e else None
    for e in uses_e_rezept
]

prescriptions = pd.DataFrame({
    "prescription_id": np.arange(1, n_rx + 1),
    "order_id": rx_orders_df["order_id"].values,
    "e_rezept_token": e_rezept_tokens,
    "prescription_date": prescription_dates,
    "prescribing_doctor_id": np.random.randint(10000, 99999, size=n_rx),
    "is_repeat_prescription": np.random.choice([True, False], size=n_rx, p=[0.55, 0.45]),
})
prescriptions.to_csv(f"{OUT}/prescriptions.csv", index=False)
print(f"   {len(prescriptions):,} prescriptions")
print(f"      With e-Rezept token: {prescriptions['e_rezept_token'].notna().sum():,}")
print(f"      Repeat prescriptions: {prescriptions['is_repeat_prescription'].sum():,}")


# ====================
# 8. SHIPMENTS
# ====================
print(f"\n[8/9] Shipments...")

# only delivered/returned orders get a shipment record (cancelled orders never shipped)
shipped_orders = orders[orders["order_status"].isin(["Delivered", "Returned"])].copy()
n_ship = len(shipped_orders)

# DHL is the dominant carrier in DACH
carriers = np.random.choice(
    ["DHL", "Hermes", "GLS", "DPD", "Austrian_Post", "Swiss_Post"],
    size=n_ship,
    p=[0.55, 0.15, 0.10, 0.08, 0.07, 0.05]
)

# ~8% of shipments need cold chain
cold_chain_required = np.random.random(n_ship) < 0.08

# carrier reliability varies a lot - DHL best, GLS worst
# these are baseline OTIF rates from logistics benchmarks
carrier_baseline_otif = {
    "DHL":           0.86,
    "Swiss_Post":    0.83,
    "Austrian_Post": 0.81,
    "DPD":           0.78,
    "Hermes":        0.74,
    "GLS":           0.66,
}

# older FCs run smoother, newer ones still have teething issues
fc_otif_adjustment = {
    "FC001":  0.03,   # Sevenum - flagship
    "FC002":  0.01,   # Köln
    "FC003": -0.02,   # München
    "FC004": -0.04,   # Wien
    "FC005": -0.06,   # Zürich - newest
}

# get the baseline rate for each shipment based on its carrier
base_otif_rates = np.array([carrier_baseline_otif[c] for c in carriers])
fc_adjustments  = np.array([fc_otif_adjustment[fc] for fc in shipped_orders["fc_id"].values])

# cold chain is harder to deliver on time -> 15pt penalty
cold_chain_penalty = np.where(cold_chain_required, -0.15, 0)

# combined probability per shipment, clipped to a sensible range
final_otif_prob = np.clip(base_otif_rates + fc_adjustments + cold_chain_penalty, 0.30, 0.98)

# roll the dice
on_time = np.random.random(n_ship) < final_otif_prob

# on-time shipments arrive in 1-3 days, late ones 4-7
on_time_days = np.random.choice([1, 2, 3], size=n_ship, p=[0.30, 0.50, 0.20])
late_days    = np.random.choice([4, 5, 6, 7], size=n_ship, p=[0.45, 0.30, 0.15, 0.10])
delivery_days = np.where(on_time, on_time_days, late_days)

# processing time depends on order type:
#   E-Rezept Rx = digital validation, fastest (~4h)
#   Paper Rx    = manual scan + verify, slowest (~14h)
#   OTC         = no prescription to check (~4.5h)
is_rx_shipment    = shipped_orders["is_prescription_order"].values
uses_e_rezept_ship = shipped_orders["uses_e_rezept"].values

paper_rx_hours = np.random.gamma(3.5, 4, n_ship).clip(1, 72)
e_rezept_hours = np.random.gamma(2, 2, n_ship).clip(0.5, 24)
otc_hours      = np.random.gamma(1.5, 3, n_ship).clip(0.5, 48)

processing_hours = np.where(
    is_rx_shipment & uses_e_rezept_ship,
    e_rezept_hours,
    np.where(
        is_rx_shipment,
        paper_rx_hours,
        otc_hours
    )
)
processing_hours = np.round(processing_hours, 1)

shipments = pd.DataFrame({
    "shipment_id": np.arange(1, n_ship + 1),
    "order_id": shipped_orders["order_id"].values,
    "carrier": carriers,
    "processing_hours": processing_hours,
    "delivery_days": delivery_days,
    "cold_chain_required": cold_chain_required,
    "on_time_delivery": on_time,
    "shipment_cost_eur": np.round(np.random.uniform(2.5, 12, n_ship), 2),
})
shipments.to_csv(f"{OUT}/shipments.csv", index=False)
print(f"   {len(shipments):,} shipments")
print(f"      Cold chain: {shipments['cold_chain_required'].sum():,}")
print(f"      Overall OTIF: {shipments['on_time_delivery'].mean()*100:.1f}%")


# ====================
# 9. RETURNS
# ====================
print(f"\n[9/9] Returns (OTC only)...")

# Rx products are legally non-returnable in Germany, so this is OTC only
# return rates vary a lot by category - devices return most, cold meds return least
category_return_rates = {
    "Medical_Devices":      0.14,
    "Personal_Care":        0.10,
    "Baby_Mother_Care":     0.09,
    "Vitamins_Supplements": 0.07,
    "OTC_Pain_Relief":      0.05,
    "OTC_Digestive":        0.05,
    "OTC_Cold_Flu":         0.03,
}

# find every delivered OTC order
otc_delivered = orders[
    (orders["order_status"] == "Delivered") & (~orders["is_prescription_order"])
]["order_id"].values

# tag each order by the category of its first item (good enough proxy for the "main" item)
order_first_item = order_items.groupby("order_id").first().reset_index()
order_first_item = order_first_item.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)
order_category = dict(zip(order_first_item["order_id"], order_first_item["category"]))

# need this to set realistic return dates later
order_dates_map = dict(zip(orders["order_id"], pd.to_datetime(orders["order_date"])))

returns_list = []
return_id_counter = 1

for category, return_rate in category_return_rates.items():
    # filter to orders in this category
    orders_in_cat = np.array([oid for oid in otc_delivered if order_category.get(oid) == category])
    if len(orders_in_cat) == 0:
        continue

    # how many of these get returned
    n_returns_for_cat = int(len(orders_in_cat) * return_rate)
    if n_returns_for_cat == 0:
        continue

    # randomly pick which ones get returned
    returned_oids = np.random.choice(orders_in_cat, size=n_returns_for_cat, replace=False)
    reasons = np.random.choice(
        ["Wrong_Product", "Damaged_Packaging", "Expired_Item", "No_Longer_Needed",
         "Quality_Issue", "Wrong_Quantity", "Allergic_Reaction"],
        size=n_returns_for_cat,
        p=[0.22, 0.18, 0.05, 0.30, 0.12, 0.08, 0.05]
    )
    refunds = np.round(np.random.uniform(5, 80, n_returns_for_cat), 2)
    # return happens 3-30 days after the order
    day_offsets = np.random.randint(3, 30, size=n_returns_for_cat)

    for oid, reason, refund, offset in zip(returned_oids, reasons, refunds, day_offsets):
        returns_list.append({
            "return_id": return_id_counter,
            "order_id": oid,
            "return_reason": reason,
            "refund_amount_eur": refund,
            "return_date": order_dates_map[oid] + pd.Timedelta(days=int(offset)),
        })
        return_id_counter += 1

returns = pd.DataFrame(returns_list)
returns.to_csv(f"{OUT}/returns.csv", index=False)
print(f"   {len(returns):,} returns")
print(f"      Spread across {returns['order_id'].nunique():,} unique orders")


# ====================
# SUMMARY
# ====================
print("\n" + "="*60)
print("DATA GENERATION COMPLETE")
print("="*60)

summary = pd.DataFrame([
    {"Table": "fulfillment_centers", "Rows": len(fulfillment_centers)},
    {"Table": "insurance",            "Rows": len(insurance_providers)},
    {"Table": "patients",             "Rows": len(patients)},
    {"Table": "products",             "Rows": len(products)},
    {"Table": "orders",               "Rows": len(orders)},
    {"Table": "order_items",          "Rows": len(order_items)},
    {"Table": "prescriptions",        "Rows": len(prescriptions)},
    {"Table": "shipments",            "Rows": len(shipments)},
    {"Table": "returns",              "Rows": len(returns)},
])
summary["Rows"] = summary["Rows"].apply(lambda x: f"{x:,}")
print(summary.to_string(index=False))

total_rows = (len(fulfillment_centers) + len(insurance_providers) + len(patients)
              + len(products) + len(orders) + len(order_items)
              + len(prescriptions) + len(shipments) + len(returns))
print(f"\nTOTAL ROWS: {total_rows:,}")
print(f"All files saved to: ./{OUT}/")


# Quick sanity check on each table

In [ ]:
import pandas as pd
import os

OUT = "pharmaflow_data"

print("="*60)
print("VERIFICATION - File sizes + first 3 rows of each table")
print("="*60)

for f in sorted(os.listdir(OUT)):
    if f.endswith(".csv"):
        path = f"{OUT}/{f}"
        size_mb = os.path.getsize(path) / 1024 / 1024
        df = pd.read_csv(path, nrows=3)
        print(f"\n{f}  ({size_mb:.1f} MB)")
        print(f"   Columns: {list(df.columns)}")
        print(df.head(3).to_string(index=False))


# Save the data to Google Drive

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

# copy the data folder over so it survives runtime disconnects
import shutil
SRC = "pharmaflow_data"
DST = "/content/drive/MyDrive/PharmaFlow/pharmaflow_data"

import os
os.makedirs("/content/drive/MyDrive/PharmaFlow", exist_ok=True)

# wipe the destination first if it already exists
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)

print(f"Data copied to: {DST}")
print("\nFiles in Drive:")
for f in sorted(os.listdir(DST)):
    size_mb = os.path.getsize(f"{DST}/{f}") / 1024 / 1024
    print(f"   {f}  ({size_mb:.1f} MB)")
